# RSNA Knee Abnormality Detection — Turbo Mosaic 2×T4

This notebook is designed for the fastest useful first submission.

Main speed changes:

- no scan of DICOM headers for every slice;
- one selected series from each anatomical plane;
- one central file per plane;
- Sagittal, Coronal and Axial images are packed into the three channels of one image;
- one image token per study instead of twelve 2.5D tokens;
- only the first four pretrained DINOv2-small blocks are used;
- the frozen encoder is distributed over both T4 GPUs;
- one final study model is trained on all exact and report-derived labels;
- no cross-validation ensemble in the default turbo configuration.

The trade-off is lower spatial coverage than the stronger notebook, but feature extraction should be dramatically faster.

In [1]:
import os, gc, re, json, math, time, random, warnings, unicodedata
from copy import deepcopy
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import pydicom

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True


class CFG:
    seed = 2026
    debug = False

    # Ultra-fast image extraction
    img_size = 112
    n_slices = 1
    n_slots = 1
    dino_layers = 4
    feature_batch_size = 512
    feature_loader_batch_size = 128
    feature_workers = 8  # threads, not multiprocessing workers
    feature_tta_hflip = False
    rebuild_features = False
    feature_cache_tag = "turbo_mosaic_dino112_l4_threaded_v2"

    # Tiny study-level classifier
    hidden_dim = 128
    dropout = 0.12
    train_batch_size = 512
    epochs = 8
    steps_per_epoch = 24
    lr = 8e-4
    weight_decay = 2e-4
    rank_loss_weight = 0.03
    exact_sample_boost = 20.0
    ema_decay = 0.995

    # Pseudo-label confidence
    rule_threshold = 0.15
    text_threshold = 0.32
    min_pseudo_weight = 0.20
    max_pseudo_weight = 0.85

    competition_dir = None
    dinov2_dir = None

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    amp = torch.cuda.is_available()
    n_gpus = torch.cuda.device_count()


TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

PLANES = ["Sagittal", "Coronal", "Axial"]
SLOT_NAMES = ["Sagittal", "Coronal", "Axial"]


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def find_competition_dir():
    if CFG.competition_dir is not None:
        p = Path(CFG.competition_dir)
        if p.exists():
            return p
    expected = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
    if expected.exists():
        return expected
    roots = [Path("/kaggle/input"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        for train_csv in root.glob("*/train.csv"):
            p = train_csv.parent
            required = [
                p / "train_series.csv", p / "test.csv", p / "test_series.csv",
                p / "train_series", p / "test_series", p / "sample_submission.csv"
            ]
            if all(x.exists() for x in required):
                return p
    raise FileNotFoundError("Competition directory was not found.")


def find_dinov2_dir():
    if CFG.dinov2_dir is not None:
        p = Path(CFG.dinov2_dir)
        if (p / "config.json").exists():
            return p

    search_root = Path("/kaggle/input")
    skip_dirs = {"train_series", "test_series", ".git", "__pycache__"}
    candidates = []
    if search_root.exists():
        base_depth = len(search_root.parts)
        for root, dirs, files in os.walk(search_root):
            depth = len(Path(root).parts) - base_depth
            dirs[:] = [d for d in dirs if d not in skip_dirs and depth < 7]
            if "config.json" not in files:
                continue
            root_path = Path(root)
            has_weights = any(
                (root_path / name).exists()
                for name in ("model.safetensors", "pytorch_model.bin")
            )
            if not has_weights:
                continue
            try:
                cfg = json.loads((root_path / "config.json").read_text())
            except Exception:
                continue
            model_type = str(cfg.get("model_type", "")).lower()
            name_hint = str(root_path).lower()
            if "dinov2" in model_type or "dinov2" in name_hint:
                hidden = int(cfg.get("hidden_size", 10_000))
                candidates.append((abs(hidden - 384), root_path))
    if not candidates:
        raise FileNotFoundError(
            "DINOv2 weights were not found. Add facebook/dinov2-small as a Kaggle model input "
            "or set CFG.dinov2_dir to its local folder."
        )
    return sorted(candidates, key=lambda x: x[0])[0][1]


seed_everything(CFG.seed)
BASE = find_competition_dir()
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

print("Device:", CFG.device)
print("Visible GPUs:", CFG.n_gpus, [torch.cuda.get_device_name(i) for i in range(CFG.n_gpus)])
print("Competition:", BASE)

Device: cuda:0
Visible GPUs: 2 ['Tesla T4', 'Tesla T4']
Competition: /kaggle/input/competitions/rsna-knee-abnormality-detection


In [2]:
train = pd.read_csv(BASE / "train.csv")
train_series = pd.read_csv(BASE / "train_series.csv")
test = pd.read_csv(BASE / "test.csv")
test_series = pd.read_csv(BASE / "test_series.csv")
sample_submission = pd.read_csv(BASE / "sample_submission.csv")


def normalize_column_names(df):
    df = df.copy()
    df.columns = [str(col).replace("\ufeff", "").strip() for col in df.columns]

    canonical = {
        "studyinstanceuid": "StudyInstanceUID",
        "patientsex": "PatientSex",
        "report": "Report",
        "acl": "ACL",
        "mcl": "MCL",
        "medial meniscus": "Medial Meniscus",
        "lateral meniscus": "Lateral Meniscus",
        "medial oa": "Medial OA",
        "lateral oa": "Lateral OA",
        "pf oa": "PF OA",
        "effusion": "Effusion",
        "synovitis": "Synovitis",
        "baker's": "Baker's",
        "bakers": "Baker's",
        "contusion": "Contusion",
        "fracture": "Fracture",
    }

    rename_map = {}
    for col in df.columns:
        key = re.sub(r"\s+", " ", col.lower()).strip()
        if key in canonical and canonical[key] not in df.columns:
            rename_map[col] = canonical[key]

    return df.rename(columns=rename_map)


train = normalize_column_names(train)
test = normalize_column_names(test)
train_series = normalize_column_names(train_series)
test_series = normalize_column_names(test_series)
sample_submission = normalize_column_names(sample_submission)

required_train_columns = ["StudyInstanceUID", "Report"] + TARGETS
missing_train_columns = [col for col in required_train_columns if col not in train.columns]
if missing_train_columns:
    raise KeyError(
        "Missing required columns in train.csv: "
        + ", ".join(missing_train_columns)
        + f"\nAvailable columns: {train.columns.tolist()}"
    )

for df in (train, test):
    if "PatientSex" not in df.columns:
        df["PatientSex"] = ""

for col in TARGETS:
    train[col] = pd.to_numeric(train[col], errors="coerce")

exact_mask = train[TARGETS].notna().values
fully_labeled_mask = exact_mask.all(axis=1)
any_labeled_mask = exact_mask.any(axis=1)

print(f"Train studies: {len(train):,}")
print(f"Test studies:  {len(test):,}")
print(f"Fully labeled studies: {fully_labeled_mask.sum():,}")
print(f"Studies with at least one exact label: {any_labeled_mask.sum():,}")
print(f"Train series: {len(train_series):,}")
print(f"Test series:  {len(test_series):,}")
print("Optional PatientSex present in source:", train["PatientSex"].astype(str).str.len().gt(0).any())
print("Train columns:", train.columns.tolist())

preview_columns = ["StudyInstanceUID", "Report"] + TARGETS
if train["PatientSex"].astype(str).str.len().gt(0).any():
    preview_columns.insert(1, "PatientSex")

display(train.loc[fully_labeled_mask, preview_columns].head(3))

Train studies: 4,407
Test studies:  3
Fully labeled studies: 58
Studies with at least one exact label: 58
Train series: 24,371
Test series:  15
Optional PatientSex present in source: False
Train columns: ['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'PatientSex']


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
29,1.2.826.0.1.3680043.8.498.10095687747295410396...,Antecedentes Clínicos:\nEsguince rodilla. [DAT...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
63,1.2.826.0.1.3680043.8.498.10170898615867673028...,The study reveals normal knee joint alignment...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
109,1.2.826.0.1.3680043.8.498.10306159113324811538...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


## 1. Multilingual report weak supervision

Absence of a phrase is treated as **unknown**, not automatically negative. Explicit positives, explicit negatives, uncertainty language and structure-level “normal/intact” statements are handled separately. The rule model is then blended with a character/word TF-IDF classifier trained only on available exact labels.

In [3]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = text.replace("’", "'").replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


NEGATION = re.compile(
    r"\b(?:no|not|without|absent|absence of|negative for|free of|"
    r"sin|ausencia de|no se observa|no se evidencia|"
    r"sem|ausencia|nao ha|nao evidencia|"
    r"kein|keine|keinen|ohne|nicht nachweisbar|"
    r"pas de|sans|absence de|aucun|aucune|"
    r"senza|assenza di|non si osserva|"
    r"geen|zonder|afwezig)\b"
)

UNCERTAIN = re.compile(
    r"\b(?:possible|possibly|probable|probably|suspect|suspected|suggestive|"
    r"cannot exclude|may represent|questionable|equivocal|"
    r"posible|probable|sugestivo|sospecha|no se descarta|"
    r"possivel|provavel|suspeita|"
    r"moglich|verdacht|"
    r"possible|probable|suspecte|"
    r"possibile|probabile|sospetto)\b"
)

NORMAL_WORDS = re.compile(
    r"\b(?:intact|normal|preserved|unremarkable|stable|"
    r"integro|integros|conservado|conservados|normal|"
    r"intacto|intactos|preservado|"
    r"intakt|unauffallig|erhalten|"
    r"intacte|intacts|normal|"
    r"integro|integri|conservato)\b"
)

INJURY_WORDS = re.compile(
    r"\b(?:tear|torn|rupture|ruptured|sprain|injury|disruption|insufficiency|"
    r"rotura|ruptura|desgarro|lesion|esguince|distension|"
    r"rottura|lesione|"
    r"dechirure|rupture|entorse|lesion|"
    r"riss|ruptur|zerrung|verletzung|"
    r"scheur|ruptuur)\w*"
)

MENISCUS_TEAR_WORDS = re.compile(
    r"\b(?:tear|torn|rupture|ruptured|fissure|cleavage|radial|bucket handle|root tear|"
    r"rotura|ruptura|desgarro|fisura|lesion|"
    r"rottura|fissurazione|lesione|"
    r"dechirure|fissure|rupture|lesion|"
    r"riss|ruptur|einriss|"
    r"scheur|ruptuur)\w*"
)

OA_WORDS = re.compile(
    r"\b(?:osteoarthrit|osteoarthros|arthros|gonarthros|degenerative joint|"
    r"cartilage loss|chondral loss|joint space narrowing|chondromalacia|"
    r"artrosis|osteoartritis|gonartrosis|degeneracion condral|condropatia|"
    r"osteoartrose|artrose|condropatia|"
    r"arthrose|gonarthrose|knorpelschaden|"
    r"arthrose|chondropathie|"
    r"artrosi|gonartrosi|condropatia)\w*"
)

ALIASES = {
    "ACL": re.compile(
        r"\b(?:acl|anterior cruciate ligament|ligamento cruzado anterior|"
        r"ligamento crociato anteriore|ligament croise anterieur|"
        r"vorderes kreuzband|voorste kruisband)\b"
    ),
    "MCL": re.compile(
        r"\b(?:mcl|medial collateral ligament|ligamento colateral medial|"
        r"ligamento collaterale mediale|ligament collateral medial|"
        r"mediales kollateralband|mediale band)\b"
    ),
    "Medial Meniscus": re.compile(
        r"\b(?:medial menisc\w*|menisc\w* medial\w*|menisco medial|"
        r"menisco interno|menisque medial|innenmeniskus|menisco mediale)\b"
    ),
    "Lateral Meniscus": re.compile(
        r"\b(?:lateral menisc\w*|menisc\w* lateral\w*|menisco lateral|"
        r"menisco externo|menisque lateral|aussenmeniskus|menisco laterale)\b"
    ),
    "Medial OA": re.compile(
        r"\b(?:medial compartment|compartimento medial|compartimento interno|"
        r"femorotibial medial|mediales kompartiment|compartiment medial)\b"
    ),
    "Lateral OA": re.compile(
        r"\b(?:lateral compartment|compartimento lateral|compartimento externo|"
        r"femorotibial lateral|laterales kompartiment|compartiment lateral)\b"
    ),
    "PF OA": re.compile(
        r"\b(?:patellofemoral|patello femoral|femoropatellar|femoro patelar|"
        r"femororotulian|retropatellar|femoropatellaire|femoropatellare)\w*"
    ),
    "Effusion": re.compile(
        r"\b(?:joint effusion|effusion|hydarthrosis|derrame articular|derrame|"
        r"liquido articular|derrame articular|erguss|gelenkerguss|"
        r"epanchement|versamento articolare)\b"
    ),
    "Synovitis": re.compile(
        r"\b(?:synovitis|sinovitis|synovite|synovial inflammation|"
        r"synoviale entzundung)\b"
    ),
    "Baker's": re.compile(
        r"\b(?:baker'?s? cyst|popliteal cyst|quiste de baker|quiste popliteo|"
        r"cisto de baker|kyste de baker|baker zyste|cisti di baker)\b"
    ),
    "Contusion": re.compile(
        r"\b(?:bone contusion|bone bruise|marrow contusion|traumatic marrow edema|"
        r"contusion osea|contusao ossea|edema ose\w*|edema de medula|"
        r"contusion osseuse|oedeme osseux|knochenmarkodem|knochenkontusion)\b"
    ),
    "Fracture": re.compile(
        r"\b(?:fracture|fractura|fratura|fraktur|frattura)\w*"
    ),
}

GROUP_NORMAL = {
    "ACL": re.compile(r"\b(?:cruciate ligaments?|ligamentos cruzados|kreuzbander)\b"),
    "MCL": re.compile(r"\b(?:collateral ligaments?|ligamentos colaterales|kollateralbander)\b"),
    "Medial Meniscus": re.compile(r"\b(?:menisci|meniscos|menisken|menisques)\b"),
    "Lateral Meniscus": re.compile(r"\b(?:menisci|meniscos|menisken|menisques)\b"),
}


def split_report(text):
    text = normalize_text(text)
    parts = re.split(r"(?<=[\.\!\?;:])\s+|\n+", text)
    return [p.strip() for p in parts if p.strip()]


def local_window(sentence, start, end, radius=90):
    lo = max(0, start - radius)
    hi = min(len(sentence), end + radius)
    return sentence[lo:hi]


def target_evidence(sentence, target):
    alias = ALIASES[target]
    evidences = []

    for m in alias.finditer(sentence):
        window = local_window(sentence, m.start(), m.end())
        negated = bool(NEGATION.search(window))
        uncertain = bool(UNCERTAIN.search(window))
        normal = bool(NORMAL_WORDS.search(window))

        if target in ("ACL", "MCL"):
            pathology = bool(INJURY_WORDS.search(window))
        elif target in ("Medial Meniscus", "Lateral Meniscus"):
            pathology = bool(MENISCUS_TEAR_WORDS.search(window))
        elif target in ("Medial OA", "Lateral OA", "PF OA"):
            pathology = bool(OA_WORDS.search(window))
        else:
            pathology = True

        if negated or (normal and not pathology):
            evidences.append((0.03, 0.95))
        elif pathology:
            if uncertain:
                evidences.append((0.68, 0.45))
            else:
                base_conf = 0.82 if target == "Contusion" else 0.95
                evidences.append((0.97, base_conf))

    if target in GROUP_NORMAL:
        group = GROUP_NORMAL[target]
        for m in group.finditer(sentence):
            window = local_window(sentence, m.start(), m.end())
            if NORMAL_WORDS.search(window) or NEGATION.search(window):
                evidences.append((0.03, 0.85))

    if target in ("Medial OA", "Lateral OA", "PF OA"):
        tri = re.search(r"\b(?:tricompartmental|tri-compartmental|tricompartimental)\b", sentence)
        if tri and OA_WORDS.search(sentence):
            evidences.append((0.97, 0.90))

    return evidences


def report_rule_scores(report):
    sentences = split_report(report)
    probs = np.full(len(TARGETS), 0.5, dtype=np.float32)
    confs = np.zeros(len(TARGETS), dtype=np.float32)

    for j, target in enumerate(TARGETS):
        candidates = []
        for i, sentence in enumerate(sentences):
            for p, c in target_evidence(sentence, target):
                # Findings near the end often occur in the impression/conclusion.
                position_bonus = 0.05 * (i / max(1, len(sentences) - 1))
                candidates.append((min(1.0, c + position_bonus), p))
        if candidates:
            candidates.sort(reverse=True)
            confs[j], probs[j] = candidates[0]
    return probs, confs


reports = train["Report"].fillna("").astype(str).tolist()
rule_probs = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
rule_conf = np.zeros_like(rule_probs)

for i, report in enumerate(tqdm(reports, desc="Rule pseudo-labels")):
    rule_probs[i], rule_conf[i] = report_rule_scores(report)

rule_rows = []
for j, target in enumerate(TARGETS):
    known = train[target].notna().values
    covered = known & (rule_conf[:, j] >= 0.50)
    if covered.sum() > 0:
        pred = (rule_probs[covered, j] >= 0.5).astype(int)
        truth = train.loc[covered, target].astype(int).values
        acc = float((pred == truth).mean())
    else:
        acc = np.nan
    rule_rows.append({
        "target": target,
        "exact_n": int(known.sum()),
        "rule_covered": int(covered.sum()),
        "rule_accuracy_on_covered": acc
    })

display(pd.DataFrame(rule_rows))

Rule pseudo-labels:   0%|          | 0/4407 [00:00<?, ?it/s]

,target,exact_n,rule_covered,rule_accuracy_on_covered
0,ACL,58,27,0.851852
1,MCL,58,27,0.851852
2,Medial Meniscus,58,31,0.870968
3,Lateral Meniscus,58,27,0.925926
4,Medial OA,58,7,0.857143
5,Lateral OA,58,6,0.666667
6,PF OA,58,11,0.909091
7,Effusion,58,38,0.684211
8,Synovitis,58,13,0.692308
9,Baker's,58,12,0.750000


In [4]:
normalized_reports = [normalize_text(x) for x in reports]

text_vectorizer = FeatureUnion([
    ("char", TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3, 5), min_df=2,
        max_features=60_000, sublinear_tf=True, dtype=np.float32
    )),
    ("word", TfidfVectorizer(
        analyzer="word", ngram_range=(1, 2), min_df=1,
        max_features=20_000, sublinear_tf=True, dtype=np.float32
    )),
])

X_text = text_vectorizer.fit_transform(normalized_reports)
print("TF-IDF shape:", X_text.shape)


def fit_text_probabilities(exclude_indices=None, compute_oof=False):
    exclude = np.zeros(len(train), dtype=bool)
    if exclude_indices is not None:
        exclude[np.asarray(exclude_indices, dtype=int)] = True

    text_probs = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
    text_oof = np.full_like(text_probs, np.nan)
    text_auc = {target: np.nan for target in TARGETS}

    for j, target in enumerate(TARGETS):
        idx = np.where(train[target].notna().values & (~exclude))[0]
        y = train.loc[idx, target].astype(int).values

        if len(idx) == 0:
            text_probs[:, j] = 0.5
            text_auc[target] = np.nan
            continue

        if len(np.unique(y)) < 2:
            text_probs[:, j] = float(y.mean())
            text_auc[target] = np.nan
            continue

        if compute_oof:
            n_splits = min(3, int(np.bincount(y).min()))
            if n_splits >= 2:
                skf = StratifiedKFold(
                    n_splits=n_splits,
                    shuffle=True,
                    random_state=CFG.seed + j
                )
                for tr_local, va_local in skf.split(idx, y):
                    tr_idx = idx[tr_local]
                    va_idx = idx[va_local]
                    model = LogisticRegression(
                        C=4.0, solver="liblinear", class_weight="balanced",
                        max_iter=2000, random_state=CFG.seed + j
                    )
                    model.fit(
                        X_text[tr_idx],
                        train.loc[tr_idx, target].astype(int).values
                    )
                    text_oof[va_idx, j] = model.predict_proba(X_text[va_idx])[:, 1]

                valid_oof = ~np.isnan(text_oof[idx, j])
                if valid_oof.sum() and len(np.unique(y[valid_oof])) == 2:
                    text_auc[target] = roc_auc_score(
                        y[valid_oof],
                        text_oof[idx[valid_oof], j]
                    )
                else:
                    text_auc[target] = np.nan
            else:
                text_auc[target] = np.nan

        final_model = LogisticRegression(
            C=4.0, solver="liblinear", class_weight="balanced",
            max_iter=2000, random_state=CFG.seed + j
        )
        final_model.fit(X_text[idx], y)
        text_probs[:, j] = final_model.predict_proba(X_text)[:, 1]

    return text_probs, text_oof, text_auc


def make_supervision(exclude_indices=None, compute_oof=False):
    text_probs, text_oof, text_auc = fit_text_probabilities(
        exclude_indices=exclude_indices,
        compute_oof=compute_oof
    )

    rule_mix = (0.25 + 0.65 * rule_conf).clip(0.25, 0.90)
    pseudo_probs = rule_mix * rule_probs + (1.0 - rule_mix) * text_probs

    distance = np.abs(pseudo_probs - 0.5)
    pseudo_mask = (
        ((rule_conf >= 0.55) & (distance >= CFG.rule_threshold)) |
        ((rule_conf < 0.55) & (distance >= CFG.text_threshold))
    )

    pseudo_weight = (
        CFG.min_pseudo_weight
        + 0.55 * (2.0 * distance)
        + 0.25 * rule_conf
    ).clip(CFG.min_pseudo_weight, CFG.max_pseudo_weight).astype(np.float32)

    targets = pseudo_probs.astype(np.float32)
    mask = pseudo_mask.astype(np.float32)
    weight = (pseudo_weight * mask).astype(np.float32)

    exact_values = train[TARGETS].fillna(0.5).values.astype(np.float32)
    targets[exact_mask] = exact_values[exact_mask]
    mask[exact_mask] = 1.0
    weight[exact_mask] = 1.0
    return targets, mask, weight, text_oof, text_auc


targets_all, target_mask_all, target_weight_all, text_oof, text_auc = make_supervision(
    exclude_indices=None,
    compute_oof=False
)

if any(np.isfinite(text_auc.get(target, np.nan)) for target in TARGETS):
    display(pd.DataFrame({
        "target": TARGETS,
        "text_oof_auc": [text_auc.get(target, np.nan) for target in TARGETS]
    }))
else:
    print("Text OOF AUC skipped in turbo mode (compute_oof=False).")

coverage = target_mask_all.mean(axis=0)
summary = pd.DataFrame({
    "target": TARGETS,
    "exact_count": exact_mask.sum(axis=0),
    "training_coverage": coverage,
    "soft_positive_rate": [
        np.average(
            targets_all[:, j],
            weights=np.maximum(target_weight_all[:, j], 1e-6)
        )
        for j in range(len(TARGETS))
    ]
})
display(summary)

TF-IDF shape: (4407, 80000)
Text OOF AUC skipped in turbo mode (compute_oof=False).


,target,exact_count,training_coverage,soft_positive_rate
0,ACL,58,0.316542,0.321712
1,MCL,58,0.327661,0.216773
2,Medial Meniscus,58,0.392784,0.495766
3,Lateral Meniscus,58,0.355798,0.274859
4,Medial OA,58,0.078738,0.471787
5,Lateral OA,58,0.076923,0.349905
6,PF OA,58,0.177672,0.282437
7,Effusion,58,0.518493,0.608893
8,Synovitis,58,0.103472,0.807247
9,Baker's,58,0.250964,0.456845


In [5]:
fully_idx = np.where(fully_labeled_mask)[0]
print("Fully labeled studies used in final training:", len(fully_idx))
print("Turbo mode trains one final model on all available exact and pseudo labels.")


Fully labeled studies used in final training: 58
Turbo mode trains one final model on all available exact and pseudo labels.


## 2. Turbo DICOM mosaic and truncated dual-GPU DINOv2

The previous loader opened the metadata header of every DICOM in every selected series before choosing slices. That is accurate but dominated runtime on Kaggle storage.

Turbo mode deliberately uses an approximate filename order and reads only one file from each selected plane. The three normalized grayscale planes become the R/G/B channels of one 112×112 image. This preserves all three anatomical planes while requiring only one encoder pass per study.

In [6]:
try:
    import cv2
except ImportError:
    cv2 = None


def series_score(row):
    fluid = float(row.get("Fluid_Sensitive", 0) or 0)
    fat = float(row.get("Fat_Suppression", 0) or 0)
    return 2.5 * fluid + 1.5 * fat


def build_plane_series(df_series):
    result = {}
    for study_id, rows in tqdm(
        df_series.groupby("StudyInstanceUID", sort=False),
        total=df_series["StudyInstanceUID"].nunique(),
        desc="Selecting plane series"
    ):
        study_planes = {}
        plane_values = rows["Anatomical_Plane"].astype(str).str.lower()

        for plane in PLANES:
            part = rows[plane_values == plane.lower()]
            if len(part) == 0:
                continue

            scores = part.apply(series_score, axis=1)
            best = part.loc[scores.idxmax()]
            study_planes[plane] = str(best["SeriesInstanceUID"])

        result[str(study_id)] = study_planes
    return result


def fast_middle_path(folder):
    try:
        paths = [
            Path(entry.path)
            for entry in os.scandir(folder)
            if entry.is_file() and entry.name.lower().endswith(".dcm")
        ]
    except Exception:
        return None

    if not paths:
        return None

    paths.sort(key=lambda p: p.name)
    return paths[len(paths) // 2]


def read_dicom_plane(path):
    if path is None:
        return None

    try:
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
    except Exception:
        return None

    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
        arr = arr.max() - arr

    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return None

    nonzero = finite[np.abs(finite) > 1e-8]
    source = nonzero if nonzero.size >= 64 else finite
    lo, hi = np.percentile(source, [1.0, 99.0])

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(finite.min()), float(finite.max())

    arr = np.nan_to_num(arr, nan=lo, posinf=hi, neginf=lo)
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / max(hi - lo, 1e-6)
    return arr.astype(np.float32)


def crop_resize_plane(arr, size):
    mask = arr > 0.03
    h, w = arr.shape

    if mask.sum() > 0.02 * h * w:
        ys, xs = np.where(mask)
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        my = max(2, int(0.06 * (y1 - y0)))
        mx = max(2, int(0.06 * (x1 - x0)))
        arr = arr[
            max(0, y0 - my):min(h, y1 + my),
            max(0, x0 - mx):min(w, x1 + mx)
        ]

    h, w = arr.shape
    side = max(h, w)
    padded = np.zeros((side, side), dtype=np.float32)
    oy = (side - h) // 2
    ox = (side - w) // 2
    padded[oy:oy + h, ox:ox + w] = arr

    if cv2 is not None:
        interpolation = cv2.INTER_AREA if side > size else cv2.INTER_LINEAR
        resized = cv2.resize(padded, (size, size), interpolation=interpolation)
    else:
        tensor = torch.from_numpy(padded)[None, None]
        resized = F.interpolate(
            tensor, size=(size, size), mode="bilinear", align_corners=False
        )[0, 0].numpy()

    return np.clip(resized * 255.0, 0, 255).astype(np.uint8)


class StudyDicomDataset(Dataset):
    def __init__(self, studies, plane_series, series_root):
        self.studies = studies.reset_index(drop=True)
        self.plane_series = plane_series
        self.series_root = Path(series_root)

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, idx):
        row = self.studies.iloc[idx]
        study_id = str(row["StudyInstanceUID"])
        selected = self.plane_series.get(study_id, {})

        mosaic = np.zeros((3, CFG.img_size, CFG.img_size), dtype=np.uint8)
        valid_planes = 0

        for channel, plane in enumerate(PLANES):
            series_id = selected.get(plane)
            if series_id is None:
                continue

            folder = self.series_root / study_id / series_id
            path = fast_middle_path(folder)
            arr = read_dicom_plane(path)
            if arr is None:
                continue

            mosaic[channel] = crop_resize_plane(arr, CFG.img_size)
            valid_planes += 1

        images = mosaic[None, None]
        token_mask = np.asarray([[valid_planes > 0]], dtype=np.bool_)

        sex = str(row.get("PatientSex", "")).strip().lower()
        sex_id = 1 if sex.startswith("m") else 2 if sex.startswith("f") else 0

        return (
            torch.from_numpy(images),
            torch.from_numpy(token_mask),
            torch.tensor(sex_id, dtype=torch.long),
            study_id
        )


train_slots = build_plane_series(train_series)
test_slots = build_plane_series(test_series)

plane_counts = {
    plane: sum(plane in item for item in train_slots.values())
    for plane in PLANES
}
display(pd.DataFrame({
    "plane": list(plane_counts),
    "train_studies_with_plane": list(plane_counts.values())
}))


Selecting plane series:   0%|          | 0/4407 [00:00<?, ?it/s]

Selecting plane series:   0%|          | 0/3 [00:00<?, ?it/s]

,plane,train_studies_with_plane
0,Sagittal,4407
1,Coronal,4407
2,Axial,4407


In [7]:
from transformers import AutoModel


class DinoCLS(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        return out.last_hidden_state[:, 0]


def load_dinov2():
    model_dir = find_dinov2_dir()
    backbone = AutoModel.from_pretrained(
        str(model_dir),
        local_files_only=True,
        trust_remote_code=False
    )
    hidden = int(backbone.config.hidden_size)

    original_layers = None
    if hasattr(backbone, "encoder") and hasattr(backbone.encoder, "layer"):
        original_layers = len(backbone.encoder.layer)
        keep = min(CFG.dino_layers, original_layers)
        backbone.encoder.layer = nn.ModuleList(
            list(backbone.encoder.layer)[:keep]
        )
        backbone.config.num_hidden_layers = keep

    for p in backbone.parameters():
        p.requires_grad = False

    model = DinoCLS(backbone).eval().to(CFG.device)
    if CFG.n_gpus > 1:
        model = nn.DataParallel(model, device_ids=list(range(CFG.n_gpus)))

    print("DINOv2:", model_dir)
    print("Hidden size:", hidden)
    print("DINO blocks:", CFG.dino_layers, "/", original_layers)
    print("Feature extraction GPUs:", max(1, CFG.n_gpus))
    print("Turbo input:", CFG.img_size, "one 3-plane mosaic per study")
    return model, hidden


@torch.inference_mode()
def encode_chunks(model, images):
    mean = torch.tensor(
        [0.485, 0.456, 0.406], device=CFG.device
    ).view(1, 3, 1, 1)
    std = torch.tensor(
        [0.229, 0.224, 0.225], device=CFG.device
    ).view(1, 3, 1, 1)

    outputs = []
    for start in range(0, len(images), CFG.feature_batch_size):
        x = images[start:start + CFG.feature_batch_size].to(
            CFG.device, non_blocking=True
        ).float().div_(255.0)
        x = (x - mean) / std

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=CFG.amp
        ):
            feat = model(x)

        outputs.append(feat.float().cpu())
    return torch.cat(outputs, dim=0)


def extract_features(studies, slots, series_root, prefix, model, hidden_size):
    stem = f"{prefix}_{CFG.feature_cache_tag}"
    feat_path = WORK / f"{stem}_features.npy"
    mask_path = WORK / f"{stem}_token_mask.npy"
    sex_path = WORK / f"{stem}_sex.npy"
    ids_path = WORK / f"{stem}_ids.npy"
    done_path = WORK / f"{stem}.done"

    feature_dim = hidden_size
    expected_shape = (len(studies), CFG.n_slots, CFG.n_slices, feature_dim)
    expected_mask_shape = expected_shape[:-1]

    cache_files = (feat_path, mask_path, sex_path, ids_path)
    cache_ok = all(p.exists() for p in cache_files) and done_path.exists()

    if cache_ok and not CFG.rebuild_features:
        try:
            cached = np.load(feat_path, mmap_mode="r")
            cached_mask = np.load(mask_path, mmap_mode="r")
            cached_sex = np.load(sex_path, mmap_mode="r")
            cached_ids = np.load(ids_path, allow_pickle=True)

            valid_cache = (
                cached.shape == expected_shape
                and cached_mask.shape == expected_mask_shape
                and cached_sex.shape == (len(studies),)
                and len(cached_ids) == len(studies)
            )
            if valid_cache:
                print(f"Using completed cached {prefix} features:", cached.shape)
                return feat_path, mask_path, sex_path, ids_path
        except Exception as exc:
            print(f"Ignoring invalid {prefix} cache:", repr(exc))

    # A shape-correct .npy can still be incomplete after a crashed DataLoader.
    # Only the .done marker makes a cache reusable.
    done_path.unlink(missing_ok=True)

    ds = StudyDicomDataset(studies, slots, series_root)
    workers = max(1, min(CFG.feature_workers, os.cpu_count() or CFG.feature_workers))
    batch_size = CFG.feature_loader_batch_size

    print(
        f"{prefix}: thread-based DICOM loading "
        f"({workers} threads, no multiprocessing queues)"
    )

    features = np.lib.format.open_memmap(
        feat_path, mode="w+", dtype=np.float16, shape=expected_shape
    )
    masks = np.lib.format.open_memmap(
        mask_path, mode="w+", dtype=np.bool_, shape=expected_mask_shape
    )
    sexes = np.lib.format.open_memmap(
        sex_path, mode="w+", dtype=np.int8, shape=(len(studies),)
    )
    ids = np.empty(len(studies), dtype=object)

    def safe_get(index):
        try:
            return ds[index]
        except Exception as exc:
            row = studies.iloc[index]
            study_id = str(row["StudyInstanceUID"])
            images = torch.zeros(
                CFG.n_slots, CFG.n_slices, 3,
                CFG.img_size, CFG.img_size,
                dtype=torch.uint8
            )
            token_mask = torch.zeros(
                CFG.n_slots, CFG.n_slices, dtype=torch.bool
            )
            return images, token_mask, torch.tensor(0, dtype=torch.long), study_id

    cursor = 0
    starts = range(0, len(ds), batch_size)

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for start in tqdm(
            starts,
            total=math.ceil(len(ds) / batch_size),
            desc=f"{prefix} threaded turbo features"
        ):
            stop = min(start + batch_size, len(ds))
            batch_indices = list(range(start, stop))

            if workers == 1:
                items = [safe_get(i) for i in batch_indices]
            else:
                items = list(pool.map(safe_get, batch_indices))

            images = torch.stack([item[0] for item in items], dim=0)
            token_mask = torch.stack([item[1] for item in items], dim=0)
            sex_id = torch.stack([item[2] for item in items], dim=0)
            study_ids = [item[3] for item in items]

            b, s, n = token_mask.shape
            valid_flat = token_mask.reshape(-1).bool()
            batch_feat = torch.zeros(
                b * s * n, feature_dim, dtype=torch.float32
            )

            if valid_flat.any():
                flat_images = images.reshape(
                    b * s * n, 3, CFG.img_size, CFG.img_size
                )
                encoded = encode_chunks(model, flat_images[valid_flat])
                batch_feat[valid_flat] = encoded

            end = cursor + b
            features[cursor:end] = batch_feat.reshape(
                b, CFG.n_slots, CFG.n_slices, feature_dim
            ).numpy().astype(np.float16)
            masks[cursor:end] = token_mask.numpy()
            sexes[cursor:end] = sex_id.numpy().astype(np.int8)
            ids[cursor:end] = np.asarray(
                [str(x) for x in study_ids], dtype=object
            )
            cursor = end

            if CFG.debug and cursor >= 32:
                break

    features.flush()
    masks.flush()
    sexes.flush()
    np.save(ids_path, ids)

    del features, masks, sexes

    if cursor == len(studies):
        done_path.write_text(
            f"rows={cursor}\nshape={expected_shape}\n",
            encoding="utf-8"
        )
        print(f"Completed {prefix} feature cache:", expected_shape)
    else:
        print(
            f"{prefix} cache is partial ({cursor}/{len(studies)}) "
            "and will not be reused."
        )

    return feat_path, mask_path, sex_path, ids_path


dinov2, dinov2_hidden = load_dinov2()


train_feature_files = extract_features(
    train, train_slots, BASE / "train_series",
    "train", dinov2, dinov2_hidden
)
test_feature_files = extract_features(
    test, test_slots, BASE / "test_series",
    "test", dinov2, dinov2_hidden
)

del dinov2
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

DINOv2: /kaggle/input/models/metaresearch/dinov2/pytorch/small/1
Hidden size: 384
DINO blocks: 4 / 12
Feature extraction GPUs: 2
Turbo input: 112 one 3-plane mosaic per study
train: thread-based DICOM loading (4 threads, no multiprocessing queues)


train threaded turbo features:   0%|          | 0/35 [00:00<?, ?it/s]

Completed train feature cache: (4407, 1, 1, 384)
test: thread-based DICOM loading (4 threads, no multiprocessing queues)


test threaded turbo features:   0%|          | 0/1 [00:00<?, ?it/s]

Completed test feature cache: (3, 1, 1, 384)


## 3. Tiny study MLP

There is only one pretrained feature vector per study, so target-token attention is unnecessary. A small normalized MLP predicts all twelve labels. Exact labels are strongly oversampled, while report-derived soft labels provide broad weak supervision.

In [8]:
class FeatureDataset(Dataset):
    def __init__(
        self, feature_files, indices,
        labels=None, label_mask=None, label_weight=None,
        exact_cells=None, train_mode=False
    ):
        feat_path, mask_path, sex_path, ids_path = feature_files
        self.features = np.load(feat_path, mmap_mode="r")
        self.token_mask = np.load(mask_path, mmap_mode="r")
        self.sex = np.load(sex_path, mmap_mode="r")
        self.ids = np.load(ids_path, allow_pickle=True)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = labels
        self.label_mask = label_mask
        self.label_weight = label_weight
        self.exact_cells = exact_cells
        self.train_mode = train_mode

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        feat = torch.from_numpy(
            np.asarray(self.features[idx], dtype=np.float32).copy()
        )
        token_mask = torch.from_numpy(
            np.asarray(self.token_mask[idx], dtype=np.bool_).copy()
        )
        sex = torch.tensor(int(self.sex[idx]), dtype=torch.long)

        if self.train_mode:
            y = torch.from_numpy(self.labels[idx].astype(np.float32))
            ym = torch.from_numpy(self.label_mask[idx].astype(np.float32))
            yw = torch.from_numpy(self.label_weight[idx].astype(np.float32))
            ex = torch.from_numpy(self.exact_cells[idx].astype(np.bool_))
            return feat, token_mask, sex, y, ym, yw, ex, idx

        return feat, token_mask, sex, str(self.ids[idx]), idx


class FastStudyMLP(nn.Module):
    def __init__(self, feature_dim, n_targets=len(TARGETS)):
        super().__init__()
        h = CFG.hidden_dim
        self.norm = nn.LayerNorm(feature_dim)
        self.sex_embed = nn.Embedding(3, h)
        self.backbone = nn.Sequential(
            nn.Linear(feature_dim, h),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
            nn.Linear(h, h),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.head = nn.Linear(h, n_targets)

    def forward(self, features, token_mask, sex):
        x = features[:, 0, 0]
        valid = token_mask[:, 0, 0].float().unsqueeze(1)
        x = self.backbone(self.norm(x))
        x = x + self.sex_embed(sex)
        x = x * valid
        return self.head(x)


def compute_pos_weight(indices, targets, label_mask, label_weight):
    y = targets[indices]
    m = label_mask[indices]
    w = label_weight[indices]
    pos = (y * m * w).sum(axis=0)
    neg = ((1.0 - y) * m * w).sum(axis=0)
    pw = np.sqrt((neg + 1.0) / (pos + 1.0))
    return torch.tensor(np.clip(pw, 0.75, 4.0), dtype=torch.float32, device=CFG.device)


def masked_bce(logits, targets, mask, weight, pos_weight):
    loss = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    effective = mask * weight
    return (loss * effective).sum() / effective.sum().clamp_min(1.0)


def pairwise_ranking_loss(logits, targets, exact_cells):
    losses = []
    hard = targets > 0.5
    for j in range(logits.shape[1]):
        valid = exact_cells[:, j]
        pos = logits[valid & hard[:, j], j]
        neg = logits[valid & (~hard[:, j]), j]
        if len(pos) and len(neg):
            diff = pos[:, None] - neg[None, :]
            losses.append(F.softplus(-diff).mean())
    if not losses:
        return logits.new_tensor(0.0)
    return torch.stack(losses).mean()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, truths = [], []
    for feat, tm, sex, y, _, _, _, _ in loader:
        feat = feat.to(CFG.device, non_blocking=True)
        tm = tm.to(CFG.device, non_blocking=True)
        sex = sex.to(CFG.device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG.amp):
            logits = model(feat, tm, sex)
        preds.append(torch.sigmoid(logits).float().cpu().numpy())
        truths.append(y.numpy())
    p = np.concatenate(preds)
    y = np.concatenate(truths)
    aucs = {}
    for j, target in enumerate(TARGETS):
        if len(np.unique(y[:, j])) == 2:
            aucs[target] = roc_auc_score(y[:, j], p[:, j])
        else:
            aucs[target] = np.nan
    macro = float(np.nanmean(list(aucs.values())))
    return macro, aucs, p


@torch.no_grad()
def update_ema(ema_model, model, decay):
    ema_params = dict(ema_model.named_parameters())
    model_params = dict(model.named_parameters())
    for name, p in model_params.items():
        ema_params[name].mul_(decay).add_(p.detach(), alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    model_buffers = dict(model.named_buffers())
    for name, b in model_buffers.items():
        ema_buffers[name].copy_(b)


feature_dim = dinov2_hidden
print("Study token feature dimension:", feature_dim)

Study token feature dimension: 384


In [9]:
all_indices = np.arange(len(train))
exact_sample = exact_mask.any(axis=1)

train_ds = FeatureDataset(
    train_feature_files,
    all_indices,
    labels=targets_all,
    label_mask=target_mask_all,
    label_weight=target_weight_all,
    exact_cells=exact_mask,
    train_mode=True
)

sample_weights = np.ones(len(all_indices), dtype=np.float64)
sample_weights[exact_sample] = CFG.exact_sample_boost

sampler = WeightedRandomSampler(
    sample_weights,
    num_samples=max(len(all_indices), CFG.train_batch_size * CFG.steps_per_epoch),
    replacement=True,
    generator=torch.Generator().manual_seed(CFG.seed)
)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.train_batch_size,
    sampler=sampler,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

model = FastStudyMLP(feature_dim).to(CFG.device)
ema_model = deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay
)

total_steps = CFG.epochs * len(train_loader)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=CFG.lr,
    total_steps=total_steps,
    pct_start=0.15,
    anneal_strategy="cos",
    div_factor=10.0,
    final_div_factor=30.0
)

scaler = torch.cuda.amp.GradScaler(enabled=CFG.amp)
pos_weight = compute_pos_weight(
    all_indices,
    targets_all,
    target_mask_all,
    target_weight_all
)

for epoch in range(1, CFG.epochs + 1):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader):
        feat, tm, sex, y, ym, yw, ex, _ = batch
        feat = feat.to(CFG.device, non_blocking=True)
        tm = tm.to(CFG.device, non_blocking=True)
        sex = sex.to(CFG.device, non_blocking=True)
        y = y.to(CFG.device, non_blocking=True)
        ym = ym.to(CFG.device, non_blocking=True)
        yw = yw.to(CFG.device, non_blocking=True)
        ex = ex.to(CFG.device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG.amp):
            logits = model(feat, tm, sex)
            bce = masked_bce(logits, y, ym, yw, pos_weight)
            rank = pairwise_ranking_loss(logits, y, ex)
            loss = bce + CFG.rank_loss_weight * rank

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        update_ema(ema_model, model, CFG.ema_decay)

        running += float(loss.item())

        if CFG.debug and step >= 2:
            break

    print(
        f"Epoch {epoch:02d}/{CFG.epochs} "
        f"| loss={running / max(1, step + 1):.4f} "
        f"| lr={optimizer.param_groups[0]['lr']:.2e}"
    )

model_path = WORK / "knee_turbo_mosaic_mlp.pt"
best_state = {
    key: value.detach().cpu().clone()
    for key, value in ema_model.state_dict().items()
}

torch.save({
    "state_dict": best_state,
    "feature_dim": feature_dim,
    "targets": TARGETS,
    "cfg": {
        "img_size": CFG.img_size,
        "dino_layers": CFG.dino_layers,
        "feature_cache_tag": CFG.feature_cache_tag
    }
}, model_path)

fold_models = [model_path]
print("Saved:", model_path)

del model, ema_model, optimizer, scheduler, scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Epoch 01/8 | loss=0.7583 | lr=7.67e-04
Epoch 02/8 | loss=0.7387 | lr=7.70e-04
Epoch 03/8 | loss=0.7394 | lr=6.64e-04
Epoch 04/8 | loss=0.7357 | lr=5.03e-04
Epoch 05/8 | loss=0.7343 | lr=3.21e-04
Epoch 06/8 | loss=0.7281 | lr=1.55e-04
Epoch 07/8 | loss=0.7284 | lr=4.11e-05
Epoch 08/8 | loss=0.7278 | lr=2.74e-06
Saved: /kaggle/working/knee_turbo_mosaic_mlp.pt


In [10]:
@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    logits_all, ids_all = [], []
    for feat, tm, sex, study_ids, _ in tqdm(loader, leave=False):
        feat = feat.to(CFG.device, non_blocking=True)
        tm = tm.to(CFG.device, non_blocking=True)
        sex = sex.to(CFG.device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=CFG.amp):
            logits = model(feat, tm, sex)
        logits_all.append(logits.float().cpu().numpy())
        ids_all.extend(list(study_ids))
    return np.concatenate(logits_all), ids_all


test_ds = FeatureDataset(
    test_feature_files,
    np.arange(len(test)),
    train_mode=False
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.train_batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

fold_logits = []
prediction_ids = None

for fold_path in fold_models:
    checkpoint = torch.load(fold_path, map_location="cpu")
    model = FastStudyMLP(checkpoint["feature_dim"]).to(CFG.device)
    model.load_state_dict(checkpoint["state_dict"])
    logits, ids = predict_model(model, test_loader)
    fold_logits.append(logits)
    if prediction_ids is None:
        prediction_ids = ids
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

mean_logits = np.mean(fold_logits, axis=0)
test_probs = 1.0 / (1.0 + np.exp(-mean_logits))

pred_df = pd.DataFrame(test_probs, columns=TARGETS)
pred_df.insert(0, "StudyInstanceUID", prediction_ids)

submission = sample_submission[["StudyInstanceUID"]].merge(
    pred_df, on="StudyInstanceUID", how="left"
)
for target in TARGETS:
    submission[target] = submission[target].fillna(0.5).clip(1e-5, 1 - 1e-5)

submission.to_csv(WORK / "submission.csv", index=False)
submission.to_csv("submission.csv", index=False)

print("submission.csv:", submission.shape)
display(submission.head())

  0%|          | 0/1 [00:00<?, ?it/s]

submission.csv: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.511396,0.378574,0.484350,0.435481,0.367951,0.425128,0.411347,0.553747,0.491410,0.407746,0.414129,0.460259
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.507377,0.382891,0.484815,0.441041,0.379608,0.426979,0.413122,0.550758,0.498687,0.409516,0.416441,0.459713
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.504669,0.388966,0.476793,0.431884,0.367157,0.414780,0.398582,0.545437,0.493794,0.405508,0.423518,0.458349


## Runtime profile

The slow v3 pipeline used 12 image tokens per study, and every token required a 2.5D triplet. Before that, it opened the headers of every DICOM in each selected series to recover physical ordering.

Turbo v4 instead performs:

- three DICOM pixel decodes per study;
- zero full-series header scans;
- one encoder image per study;
- four DINOv2 blocks instead of twelve;
- one final MLP instead of a three-fold ensemble.

This is intended as a rapid iteration baseline. The next quality step is to increase `dino_layers` to 6 or use two mosaics per study after confirming the pipeline and submission format.

## Stability update in v4.2

Kaggle's Python 3.12 environment can occasionally corrupt multiprocessing
`DataLoader` queues, producing `Bad file descriptor` and semaphore errors.
This version uses `ThreadPoolExecutor` for parallel DICOM I/O and
`num_workers=0` for the lightweight memmap loaders. No multiprocessing queue,
semaphore, or persistent worker is created.

Feature caches are accepted only when a matching `.done` marker exists, so a
partially written cache from an interrupted run cannot silently be reused.
